# Final Project - ST 554
Author: Max Campbell

In this assignment, we will demonstrate the capabilities of building models via pySpark using machine learning tools and data streaming! In particular, we want to build a pipeline that we can use to fit data to an elastic net model (chosen because we can cross-validate to find the optimal tuning parameters, which in turn will allow the model to keep itself relatively stable at high levels of complexity), and use that pipeline to predict new data against the model quickly and effectively. Let's begin by reading in the base dataset that we will use to fit the model. Our dataset of choice is power readings from Tetouan, Morocco as it relates to various environmental factors such as temperature, humidity, and time of day. The variable of interest is `Power_Zone_3`, representing power readings from a subsection of the city. In this context, we imagine that we are anticipating the tools used to measure `Power_Zone_3` are going offline soon, and we want a way to predict the power output while the tools are offline. Let's go ahead and get started!

In [1]:
#Load in necessary modules
import pandas as pd
from pyspark.sql import SparkSession

#Read in data as a pandas DataFrame
power = pd.read_csv("power_ml_data.csv")

#Initialize Spark session
spark = SparkSession.builder.master('local[*]').appName('FP') \
    .config("spark.sql.ansi.enabled", "false").getOrCreate()

#Read pandas DF into Spark
power = spark.createDataFrame(power)

power.show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/26 10:34:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
|      5.853|    76.9|     0.081|       

Now that we've got our data in Spark, it's time to start setting up the pipeline. The transformations that we will be performing to this base dataset will also be performed on any future data that we read in, which is why using Spark/MLlib is a good tool for this job. Let's start with the `Hour` variable. We want to understand whether a power reading was taken (roughly) at night or day, so we will start with binarizing this variable to create an indicator of night-time readings vs day-time readings. Note that we may also have to convert `Hour` to a `DoubleType` first to accomplish this.

In [2]:
#Check data types for the dataframe
power.printSchema()

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



In [3]:
#Hour is a long, but we want double type, so we will convert it and then binarize it
#Import necessary modules
from pyspark.ml.feature import SQLTransformer, Binarizer
from pyspark.ml import Pipeline

#Convert Hour to Double
doubleTypeConverter = SQLTransformer(statement = "SELECT *, CAST(Hour AS DOUBLE) AS Hour_d FROM __THIS__")

#Binarize Hour by whether the value is less than 6.5 or not
hourBinarizer = Binarizer(threshold = 6.5, inputCol = "Hour_d", outputCol = "isDaytime")

Next, we will need to one-hot encode the Month column so that the model can read the categorical data in a format that it can process efficiently.

In [4]:
#Import necessary modules
from pyspark.ml.feature import OneHotEncoder

#OHE Month
oneHotEncoder = OneHotEncoder(inputCols = ["Month"], outputCols = ["Month_ohe"])

After that, we will do a principal components analysis (PCA) on the environmental factors `Temperature`, `Humidity`, `Wind_Speed`, `General_Diffuse_Flows`, and `Diffuse_Flows`. This will involve doing a PCA fit on these variables, which we will then use as the transformer that fits into our overall pipeline.

In [5]:
#Import necessary modules
from pyspark.ml.feature import VectorAssembler, PCA

#Assemble PCA features into a model
pcaVecAssembler = VectorAssembler(inputCols = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"], outputCol = "pcaFeatures")

#Fit the PCA model
pca = PCA(k = 2, inputCol = "pcaFeatures", outputCol = "pcaOutputs")

pipeline = Pipeline(stages = [doubleTypeConverter, hourBinarizer, oneHotEncoder, pcaVecAssembler, pca])
testing = pipeline.fit(power)

Now we can set up for the Elastic Net model, by assembling our desired features and defining the response variable.

In [7]:
#Define label and features
labelmaker = VectorAssembler(inputCols = ["Power_Zone_3"], outputCol = "label")
vecAssembler = VectorAssembler(inputCols = ["pcaFeatures", "isDaytime", "Power_Zone_1", "Power_Zone_2", "Month_ohe"], outputCol = "features")

pipeline = Pipeline(stages = [doubleTypeConverter, hourBinarizer, oneHotEncoder, pcaVecAssembler, pca, labelmaker, vecAssembler])

[]

We are ready to cross-validate the model now!